# Bank Marketing (DP-GAN) 

Author: Ilse Harmers \
Last updated: April 21, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from snsynth import Synthesizer
import snsynth.transform as tf
import utils

In [ ]:
# Importing train set.
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")
print(bank_train.columns.to_list())
bank_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = bank_train["age"].min(), upper = bank_train["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # job
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # default
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-8000) + 1)), upper = np.log(102000 + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # housing
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # loan
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # contact
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # day
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # month
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(4900 + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = bank_train["campaign"].min(), upper = 60,
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_train["pdays"].min()) + 1)), upper = np.log(870 + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(280 + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # poutcome
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # y
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 4 * 10^4 rows, then delta = 10^(-5). 
delta = 10**np.floor(np.log10(1/bank_train.shape[0]))
print(delta)

# Defining epsilon value.
epsi = 2

# Defining differentially private GAN model: [dpgan].
model = "dpgan"

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")
    
    # Synthesizing the dataset with DP-GAN. 
    synth = Synthesizer.create(model, epsilon = epsi, delta = delta, batch_size = 512, epochs = 10000, epoch_time = True)
    synth.fit(bank_train, transformer = tt, preprocessor_eps = 0.0)

    r += 1

In [ ]:
# Saving the results on runtime.
time_per_run = pd.DataFrame({"time": [0.784869, 0.780951, 0.778333, 0.797793, 0.773469]}, index = [f"run{i}" for i in range(1, 6)])
time_per_run.to_csv(f"./Runtimes/bank_runtime_{model}.csv")
time_per_run

In [ ]:
# Computing average runtime across the runs.
print("Mean runtime [s/epoch]:", np.mean(time_per_run["time"]))
print("Standard deviation runtime [s/epoch]:", np.std(time_per_run["time"]))